In [1]:
!pip install transformers datasets accelerate scikit-learn -q

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
!pip install datasets -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")
print("Dataset loaded!")
print(dataset)

In [ ]:
# Load IMDB dataset from Hugging Face (same as Kaggle IMDB)
dataset = load_dataset("imdb")
print("Dataset loaded!")
print(dataset)

In [ ]:
import re

# Clean text function
def clean_text(example):
    text = example['text']
    text = re.sub(r'<.*?>', '', text)          # Remove HTML tags
    text = re.sub(r'http\S+', '', text)        # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)   # Keep only letters
    text = text.lower().strip()                 # Lowercase
    example['text'] = text
    return example

# Take small subset for fast training
train_data = dataset['train'].shuffle(seed=42).select(range(2000))
test_data = dataset['test'].shuffle(seed=42).select(range(500))

# Apply cleaning
train_data = train_data.map(clean_text)
test_data = test_data.map(clean_text)

# Split train into train + validation (80-20)
train_val = train_data.train_test_split(test_size=0.2, seed=42)
train_dataset = train_val['train']  # 1600 samples
val_dataset = train_val['test']     # 400 samples
test_dataset = test_data            # 500 samples

print(f"Train: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")
print(f"\nSample text: {train_dataset[0]['text'][:200]}...")
print(f"Label: {train_dataset[0]['label']} (0=Negative, 1=Positive)")

In [ ]:
# Check missing values
train_nulls = sum(1 for x in train_dataset['text'] if not x or x.isspace())
val_nulls = sum(1 for x in val_dataset['text'] if not x or x.isspace())
test_nulls = sum(1 for x in test_dataset['text'] if not x or x.isspace())

print(f"Missing/Empty - Train: {train_nulls}, Val: {val_nulls}, Test: {test_nulls}")

# Label distribution
from collections import Counter
print(f"\nTrain labels: {Counter(train_dataset['label'])}")
print(f"Val labels: {Counter(val_dataset['label'])}")
print(f"Test labels: {Counter(test_dataset['label'])}")

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("Metrics function defined!")

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ---- TOKENIZATION ----
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization complete!")

# ---- METRICS FUNCTION ----
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

# ---- EXPERIMENT 1 ----
print("=" * 60)
print("EXPERIMENT 1: Freeze ALL BERT layers, train classifier only")
print("=" * 60)

model_exp1 = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

for param in model_exp1.bert.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model_exp1.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_exp1.parameters())
print(f"Trainable: {trainable:,} / Total: {total:,} ({100*trainable/total:.2f}%)")

# KEY FIX: Added disable_tqdm=True and removed notebook progress bar issue
training_args_exp1 = TrainingArguments(
    output_dir='./results_exp1',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    disable_tqdm=False
)

trainer_exp1 = Trainer(
    model=model_exp1,
    args=training_args_exp1,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics
)

# Train
trainer_exp1.train()

# KEY FIX: Use predict() instead of evaluate() to avoid the callback error
predictions_exp1 = trainer_exp1.predict(test_tokenized)
results_exp1 = predictions_exp1.metrics

print(f"\n{'='*60}")
print("Experiment 1 Test Results:")
print(f"  Accuracy:  {results_exp1['test_accuracy']:.4f}")
print(f"  Precision: {results_exp1['test_precision']:.4f}")
print(f"  Recall:    {results_exp1['test_recall']:.4f}")
print(f"  F1 Score:  {results_exp1['test_f1']:.4f}")
print(f"{'='*60}")

In [ ]:
# Get predictions for Experiment 1
preds_exp1 = trainer_exp1.predict(test_tokenized)
pred_labels_exp1 = np.argmax(preds_exp1.predictions, axis=-1)
true_labels = preds_exp1.label_ids

# Confusion Matrix
cm1 = confusion_matrix(true_labels, pred_labels_exp1)
print("Experiment 1 - Confusion Matrix:")
print(cm1)
print("\nClassification Report:")
print(classification_report(true_labels, pred_labels_exp1, target_names=['Negative', 'Positive']))

# Plot
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm1, display_labels=['Negative', 'Positive']).plot(ax=ax, cmap='Blues')
plt.title('Experiment 1: Frozen BERT (Classifier Only)')
plt.tight_layout()
plt.savefig('confusion_matrix_exp1.png', dpi=100)
plt.show()

In [ ]:
print("=" * 60)
print("EXPERIMENT 2: Fine-tune LAST 2 BERT layers + classifier")
print("=" * 60)

# Load fresh model
model_exp2 = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Freeze ALL BERT layers first
for param in model_exp2.bert.parameters():
    param.requires_grad = False

# Unfreeze last 2 encoder layers + pooler
for param in model_exp2.bert.encoder.layer[-1].parameters():
    param.requires_grad = True
for param in model_exp2.bert.encoder.layer[-2].parameters():
    param.requires_grad = True
for param in model_exp2.bert.pooler.parameters():
    param.requires_grad = True

# Count trainable params
trainable = sum(p.numel() for p in model_exp2.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_exp2.parameters())
print(f"Trainable: {trainable:,} / Total: {total:,} ({100*trainable/total:.2f}%)")

# Training arguments
training_args_exp2 = TrainingArguments(
    output_dir='./results_exp2',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none"
)

# Trainer
trainer_exp2 = Trainer(
    model=model_exp2,
    args=training_args_exp2,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics
)

# Train
trainer_exp2.train()

# Evaluate on test set
results_exp2 = trainer_exp2.evaluate(test_tokenized)
print(f"\nExperiment 2 Results: {results_exp2}")

In [ ]:
# Get predictions for Experiment 2
preds_exp2 = trainer_exp2.predict(test_tokenized)
pred_labels_exp2 = np.argmax(preds_exp2.predictions, axis=-1)

# Confusion Matrix
cm2 = confusion_matrix(true_labels, pred_labels_exp2)
print("Experiment 2 - Confusion Matrix:")
print(cm2)
print("\nClassification Report:")
print(classification_report(true_labels, pred_labels_exp2, target_names=['Negative', 'Positive']))

# Plot
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm2, display_labels=['Negative', 'Positive']).plot(ax=ax, cmap='Greens')
plt.title('Experiment 2: Fine-tune Last 2 Layers')
plt.tight_layout()
plt.savefig('confusion_matrix_exp2.png', dpi=100)
plt.show()

In [ ]:
print("=" * 70)
print("COMPARISON OF EXPERIMENTS")
print("=" * 70)

# Create comparison table
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Trainable Params'],
    'Exp1: Frozen BERT': [
        results_exp1['eval_accuracy'],
        results_exp1['eval_precision'],
        results_exp1['eval_recall'],
        results_exp1['eval_f1'],
        '1,538 (0.00%)'
    ],
    'Exp2: Last 2 Layers': [
        results_exp2['eval_accuracy'],
        results_exp2['eval_precision'],
        results_exp2['eval_recall'],
        results_exp2['eval_f1'],
        '14.6M (13.37%)'
    ]
})

print(comparison.to_string(index=False))

# Visualization
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
exp1_scores = [results_exp1['eval_accuracy'], results_exp1['eval_precision'],
               results_exp1['eval_recall'], results_exp1['eval_f1']]
exp2_scores = [results_exp2['eval_accuracy'], results_exp2['eval_precision'],
               results_exp2['eval_recall'], results_exp2['eval_f1']]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, exp1_scores, width, label='Exp1: Frozen BERT', color='steelblue')
bars2 = ax.bar(x + width/2, exp2_scores, width, label='Exp2: Last 2 Layers', color='forestgreen')

ax.set_ylabel('Score')
ax.set_title('Experiment Comparison: BERT Fine-Tuning')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.0)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('experiment_comparison.png', dpi=100)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(cm1, display_labels=['Negative', 'Positive']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Exp 1: Frozen BERT\n(Classifier Only)', fontsize=13)

ConfusionMatrixDisplay(cm2, display_labels=['Negative', 'Positive']).plot(ax=axes[1], cmap='Greens')
axes[1].set_title('Exp 2: Fine-tune Last 2 Layers', fontsize=13)

plt.tight_layout()
plt.savefig('confusion_matrices_comparison.png', dpi=100)
plt.show()

In [17]:
print("=" * 70)
print("ANALYSIS & INSIGHTS")
print("=" * 70)

analysis = """
1. EXPERIMENT 1 - Frozen BERT (Classifier Only):
   - Only 1,538 parameters were trainable (the classification head)
   - Performance was poor (~59% accuracy) - barely above random guessing
   - BERT's frozen representations alone aren't sufficient when only 
     a simple linear classifier is trained on top
   - Training was very fast (~2 min) due to minimal gradient computation

2. EXPERIMENT 2 - Fine-tune Last 2 Layers:
   - 14.6M trainable parameters (13.37% of total)
   - Significant improvement: ~88% accuracy, ~88% F1 score
   - Fine-tuning allows BERT to adapt its representations to the 
     specific sentiment classification task
   - Training took longer (~4-5 min) but results justify the cost

3. KEY FINDINGS:
   - Fine-tuning even just 2 layers improves accuracy by ~29 percentage points
   - The last layers of BERT capture task-specific features when fine-tuned
   - Trade-off: More trainable params = better performance but slower training
   - For production: Fine-tuning last 2+ layers is recommended
   
4. DATASET:
   - Used IMDB Movie Reviews (2000 train, 400 val, 500 test)
   - Binary sentiment classification (Positive/Negative)
   - Balanced dataset with roughly equal positive and negative samples
   
5. RECOMMENDATIONS:
   - For better results: Use full dataset (25K samples)
   - Try unfreezing more layers or full fine-tuning
   - Consider DistilBERT for faster training with comparable results
"""

print(analysis)

ANALYSIS & INSIGHTS

1. EXPERIMENT 1 - Frozen BERT (Classifier Only):
   - Only 1,538 parameters were trainable (the classification head)
   - Performance was poor (~59% accuracy) - barely above random guessing
   - BERT's frozen representations alone aren't sufficient when only 
     a simple linear classifier is trained on top
   - Training was very fast (~2 min) due to minimal gradient computation

2. EXPERIMENT 2 - Fine-tune Last 2 Layers:
   - 14.6M trainable parameters (13.37% of total)
   - Significant improvement: ~88% accuracy, ~88% F1 score
   - Fine-tuning allows BERT to adapt its representations to the 
     specific sentiment classification task
   - Training took longer (~4-5 min) but results justify the cost

3. KEY FINDINGS:
   - Fine-tuning even just 2 layers improves accuracy by ~29 percentage points
   - The last layers of BERT capture task-specific features when fine-tuned
   - Trade-off: More trainable params = better performance but slower training
   - For produc

In [ ]:
print("=" * 60)
print("BONUS: DistilBERT Fine-Tuning")
print("=" * 60)

# Load DistilBERT
distil_model_name = "distilbert-base-uncased"
distil_tokenizer = AutoTokenizer.from_pretrained(distil_model_name)

# Tokenize with DistilBERT tokenizer
def tokenize_distil(examples):
    return distil_tokenizer(examples['text'], padding='max_length', 
                           truncation=True, max_length=256)

train_distil = train_dataset.map(tokenize_distil, batched=True)
val_distil = val_dataset.map(tokenize_distil, batched=True)
test_distil = test_dataset.map(tokenize_distil, batched=True)

train_distil.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_distil.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_distil.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Load model
model_distil = AutoModelForSequenceClassification.from_pretrained(distil_model_name, num_labels=2)

# Training
training_args_distil = TrainingArguments(
    output_dir='./results_distil',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none"
)

trainer_distil = Trainer(
    model=model_distil,
    args=training_args_distil,
    train_dataset=train_distil,
    eval_dataset=val_distil,
    compute_metrics=compute_metrics
)

trainer_distil.train()

# Evaluate
results_distil = trainer_distil.evaluate(test_distil)
print(f"\nDistilBERT Results: {results_distil}")

# Final comparison
print("\n" + "=" * 70)
print("FINAL COMPARISON (All 3 Models)")
print("=" * 70)
final_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Frozen BERT': [f"{results_exp1[f'eval_{m}']:.4f}" for m in ['accuracy','precision','recall','f1']],
    'BERT Last 2': [f"{results_exp2[f'eval_{m}']:.4f}" for m in ['accuracy','precision','recall','f1']],
    'DistilBERT': [f"{results_distil[f'eval_{m}']:.4f}" for m in ['accuracy','precision','recall','f1']]
})
print(final_comparison.to_string(index=False))

BONUS: DistilBERT Fine-Tuning


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
